# Phase 2.1: Target Definition
**DNA Gene Mapping Project - ML Phase**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Objective
Define clear ML prediction targets and validate their quality

## Key Tasks
1. Define binary classification targets
2. Analyze target distribution and class balance
3. Validate target quality
4. Define success metrics
5. Document target definition for downstream phases

## Deliverables
- Target definitions document
- Class distribution analysis
- Target validation report

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

PROJECT_ROOT = Path().absolute().parent.parent
REPORTS_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'reports'
FIGURES_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'figures' / 'target_definition'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

print("Setup complete")
print(f"Reports: {REPORTS_DIR}")
print(f"Figures: {FIGURES_DIR}")

In [ ]:
# Database connection
load_dotenv()

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'genome_db')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")

## 1. ML Task 1: Variant Pathogenicity Prediction

### Task Definition
**Problem Type:** Binary Classification

**Goal:** Predict whether a genetic variant is pathogenic or benign

**Target Variable:** Binary label derived from clinical_ml_features table
- Positive class (1): Pathogenic variants (target_is_pathogenic = TRUE)
- Negative class (0): Benign variants (target_is_benign = TRUE)
- Excluded: VUS (Variants of Uncertain Significance) - target_is_vus = TRUE

**Why exclude VUS?** 
- VUS have unclear clinical significance
- Including them adds noise to training
- Model should learn clear pathogenic vs benign distinction

In [ ]:
# Load target distribution from clinical_ml_features
print("Loading variant targets from clinical_ml_features...")

query = """
SELECT 
    COUNT(*) as total_variants,
    SUM(CASE WHEN target_is_pathogenic = true THEN 1 ELSE 0 END) as pathogenic_count,
    SUM(CASE WHEN target_is_benign = true THEN 1 ELSE 0 END) as benign_count,
    SUM(CASE WHEN target_is_vus = true THEN 1 ELSE 0 END) as vus_count
FROM gold.clinical_ml_features
"""

target_stats = pd.read_sql(query, engine)

total = target_stats['total_variants'].iloc[0]
pathogenic = target_stats['pathogenic_count'].iloc[0]
benign = target_stats['benign_count'].iloc[0]
vus = target_stats['vus_count'].iloc[0]

print(f"\nTotal variants: {total:,}")
print(f"  Pathogenic: {pathogenic:,} ({pathogenic/total*100:.2f}%)")
print(f"  Benign:     {benign:,} ({benign/total*100:.2f}%)")
print(f"  VUS:        {vus:,} ({vus/total*100:.2f}%)")

usable_for_training = pathogenic + benign
print(f"\nUsable for training (excluding VUS): {usable_for_training:,} ({usable_for_training/total*100:.2f}%)")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# All classes
all_classes = pd.DataFrame({
    'Class': ['Pathogenic', 'Benign', 'VUS'],
    'Count': [pathogenic, benign, vus]
})

axes[0].bar(all_classes['Class'], all_classes['Count'], 
            color=['#e74c3c', '#27ae60', '#95a5a6'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('All Variants (3 classes)', fontsize=13, fontweight='bold')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

for i, (label, count) in enumerate(zip(all_classes['Class'], all_classes['Count'])):
    axes[0].text(i, count, f'{count/total*100:.1f}%', 
                ha='center', va='bottom', fontweight='bold')

# Binary classification (excluding VUS)
binary_classes = pd.DataFrame({
    'Class': ['Pathogenic (1)', 'Benign (0)'],
    'Count': [pathogenic, benign]
})

axes[1].bar(binary_classes['Class'], binary_classes['Count'], 
            color=['#e74c3c', '#27ae60'], alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[1].set_title('ML Training Set (VUS excluded)', fontsize=13, fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

for i, (label, count) in enumerate(zip(binary_classes['Class'], binary_classes['Count'])):
    axes[1].text(i, count, f'{count/usable_for_training*100:.1f}%', 
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_variant_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {FIGURES_DIR / '01_variant_target_distribution.png'}")

In [ ]:
# Class imbalance analysis
imbalance_ratio = max(pathogenic, benign) / min(pathogenic, benign)

print(f"\nClass Imbalance Analysis:")
print(f"  Majority class: {'Benign' if benign > pathogenic else 'Pathogenic'}")
print(f"  Minority class: {'Pathogenic' if benign > pathogenic else 'Benign'}")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print(f"\n  WARNING: Significant class imbalance detected!")
    print(f"  Recommendation: Use class weights, SMOTE, or stratified sampling")
elif imbalance_ratio > 2:
    print(f"\n  NOTE: Moderate class imbalance")
    print(f"  Recommendation: Consider class weights during training")
else:
    print(f"\n  GOOD: Classes are reasonably balanced")

## 2. ML Task 2: Structural Variant Risk Prediction

### Task Definition
**Problem Type:** Binary Classification

**Goal:** Predict whether a structural variant is high-risk

**Target Variable:** Binary label from structural_variant_ml_features table
- Positive class (1): High-risk SVs (is_high_risk_sv = TRUE)
- Negative class (0): Low/medium-risk SVs (is_high_risk_sv = FALSE)

**Why this target?**
- High-risk SVs require clinical attention
- Clear binary classification problem
- Actionable predictions for clinicians

In [ ]:
# Load SV target distribution
print("Loading SV targets from structural_variant_ml_features...")

query_sv = """
SELECT 
    COUNT(*) as total_svs,
    SUM(CASE WHEN is_high_risk_sv = true THEN 1 ELSE 0 END) as high_risk_count,
    SUM(CASE WHEN is_high_risk_sv = false THEN 1 ELSE 0 END) as low_risk_count
FROM gold.structural_variant_ml_features
"""

sv_stats = pd.read_sql(query_sv, engine)

total_sv = sv_stats['total_svs'].iloc[0]
high_risk = sv_stats['high_risk_count'].iloc[0]
low_risk = sv_stats['low_risk_count'].iloc[0]

print(f"\nTotal SVs: {total_sv:,}")
print(f"  High-risk: {high_risk:,} ({high_risk/total_sv*100:.2f}%)")
print(f"  Low-risk:  {low_risk:,} ({low_risk/total_sv*100:.2f}%)")

In [ ]:
# Visualize SV target distribution
fig, ax = plt.subplots(figsize=(10, 6))

sv_classes = pd.DataFrame({
    'Class': ['High-Risk (1)', 'Low-Risk (0)'],
    'Count': [high_risk, low_risk]
})

bars = ax.bar(sv_classes['Class'], sv_classes['Count'], 
              color=['#e74c3c', '#3498db'], alpha=0.7, edgecolor='black')

ax.set_ylabel('Count', fontsize=11, fontweight='bold')
ax.set_title('Structural Variant Target Distribution', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

for i, (label, count) in enumerate(zip(sv_classes['Class'], sv_classes['Count'])):
    ax.text(i, count, f'{count/total_sv*100:.1f}%', 
            ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_sv_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {FIGURES_DIR / '02_sv_target_distribution.png'}")

In [ ]:
# SV class imbalance analysis
sv_imbalance_ratio = max(high_risk, low_risk) / min(high_risk, low_risk)

print(f"\nSV Class Imbalance Analysis:")
print(f"  Majority class: {'Low-risk' if low_risk > high_risk else 'High-risk'}")
print(f"  Minority class: {'High-risk' if low_risk > high_risk else 'Low-risk'}")
print(f"  Imbalance ratio: {sv_imbalance_ratio:.2f}:1")

if sv_imbalance_ratio > 3:
    print(f"\n  WARNING: Significant class imbalance detected!")
    print(f"  Recommendation: Use class weights, SMOTE, or stratified sampling")
elif sv_imbalance_ratio > 2:
    print(f"\n  NOTE: Moderate class imbalance")
    print(f"  Recommendation: Consider class weights during training")
else:
    print(f"\n  GOOD: Classes are reasonably balanced")

## 3. Target Validation

In [ ]:
# Check for overlapping labels in variant data
print("Validating variant targets (no overlapping labels)...")

query_overlap = """
SELECT 
    SUM(CASE WHEN target_is_pathogenic = true AND target_is_benign = true THEN 1 ELSE 0 END) as pathogenic_and_benign,
    SUM(CASE WHEN target_is_pathogenic = true AND target_is_vus = true THEN 1 ELSE 0 END) as pathogenic_and_vus,
    SUM(CASE WHEN target_is_benign = true AND target_is_vus = true THEN 1 ELSE 0 END) as benign_and_vus
FROM gold.clinical_ml_features
"""

overlap_check = pd.read_sql(query_overlap, engine)

print(f"\nOverlapping labels check:")
print(f"  Pathogenic AND Benign: {overlap_check['pathogenic_and_benign'].iloc[0]:,}")
print(f"  Pathogenic AND VUS:    {overlap_check['pathogenic_and_vus'].iloc[0]:,}")
print(f"  Benign AND VUS:        {overlap_check['benign_and_vus'].iloc[0]:,}")

total_overlap = overlap_check.sum().sum()
if total_overlap == 0:
    print(f"\n  PASS: No overlapping labels detected")
else:
    print(f"\n  WARNING: {total_overlap:,} variants have overlapping labels!")
    print(f"  Action: These will be excluded from training")

In [ ]:
# Check for missing targets
print("\nChecking for variants with no label...")

query_unlabeled = """
SELECT COUNT(*) as unlabeled_count
FROM gold.clinical_ml_features
WHERE target_is_pathogenic = false 
  AND target_is_benign = false 
  AND target_is_vus = false
"""

unlabeled = pd.read_sql(query_unlabeled, engine)['unlabeled_count'].iloc[0]

print(f"  Unlabeled variants: {unlabeled:,} ({unlabeled/total*100:.2f}%)")

if unlabeled > 0:
    print(f"  Action: These will be excluded from training")

## 4. Success Metrics Definition

In [ ]:
# Define success metrics for each task
metrics_definition = {
    'Task 1: Variant Pathogenicity': {
        'Primary Metric': 'F1-Score',
        'Reason': 'Balances precision and recall for imbalanced classes',
        'Secondary Metrics': [
            'Precision (minimize false positives)',
            'Recall (minimize false negatives)',
            'ROC-AUC (overall discrimination)',
            'PR-AUC (performance on minority class)'
        ],
        'Target Performance': {
            'Minimum acceptable F1': 0.75,
            'Good F1': 0.85,
            'Excellent F1': 0.90
        },
        'Why these thresholds?': 'Clinical relevance - must balance sensitivity and specificity'
    },
    'Task 2: SV Risk Prediction': {
        'Primary Metric': 'Recall (Sensitivity)',
        'Reason': 'Missing high-risk SVs is more costly than false alarms',
        'Secondary Metrics': [
            'Precision (control false positives)',
            'F1-Score (balance)',
            'ROC-AUC (overall discrimination)'
        ],
        'Target Performance': {
            'Minimum acceptable Recall': 0.80,
            'Good Recall': 0.90,
            'Excellent Recall': 0.95
        },
        'Why these thresholds?': 'High-risk SVs require follow-up - prefer sensitivity over specificity'
    }
}

print("ML Success Metrics Defined:")
print("="*80)
for task, metrics in metrics_definition.items():
    print(f"\n{task}")
    print("-"*80)
    print(f"  Primary Metric: {metrics['Primary Metric']}")
    print(f"  Reason: {metrics['Reason']}")
    print(f"\n  Target Performance:")
    for perf_level, value in metrics['Target Performance'].items():
        print(f"    {perf_level}: {value}")
    print(f"\n  Rationale: {metrics['Why these thresholds?']}")

## 5. Generate Target Definition Report

In [ ]:
# Generate comprehensive target definition report
report_path = REPORTS_DIR / 'target_definition_report.txt'

with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("ML TARGET DEFINITION REPORT\n")
    f.write("DNA Gene Mapping Project - Phase 2.1\n")
    f.write("="*80 + "\n\n")
    
    f.write("TASK 1: VARIANT PATHOGENICITY PREDICTION\n")
    f.write("-"*80 + "\n")
    f.write("Problem Type: Binary Classification\n")
    f.write("Target: Pathogenic (1) vs Benign (0)\n")
    f.write("Data Source: gold.clinical_ml_features\n\n")
    
    f.write("Target Distribution:\n")
    f.write(f"  Total variants: {total:,}\n")
    f.write(f"  Pathogenic:     {pathogenic:,} ({pathogenic/total*100:.2f}%)\n")
    f.write(f"  Benign:         {benign:,} ({benign/total*100:.2f}%)\n")
    f.write(f"  VUS (excluded): {vus:,} ({vus/total*100:.2f}%)\n")
    f.write(f"  Training set:   {usable_for_training:,} ({usable_for_training/total*100:.2f}%)\n\n")
    
    f.write(f"Class Imbalance: {imbalance_ratio:.2f}:1\n")
    if imbalance_ratio > 3:
        f.write("  Status: SIGNIFICANT IMBALANCE - requires class weights or SMOTE\n\n")
    elif imbalance_ratio > 2:
        f.write("  Status: MODERATE IMBALANCE - consider class weights\n\n")
    else:
        f.write("  Status: REASONABLE BALANCE - no special handling needed\n\n")
    
    f.write("Primary Metric: F1-Score\n")
    f.write("  Target: 0.85+ (Good), 0.90+ (Excellent)\n\n")
    
    f.write("\n" + "="*80 + "\n\n")
    
    f.write("TASK 2: STRUCTURAL VARIANT RISK PREDICTION\n")
    f.write("-"*80 + "\n")
    f.write("Problem Type: Binary Classification\n")
    f.write("Target: High-risk (1) vs Low-risk (0)\n")
    f.write("Data Source: gold.structural_variant_ml_features\n\n")
    
    f.write("Target Distribution:\n")
    f.write(f"  Total SVs:  {total_sv:,}\n")
    f.write(f"  High-risk:  {high_risk:,} ({high_risk/total_sv*100:.2f}%)\n")
    f.write(f"  Low-risk:   {low_risk:,} ({low_risk/total_sv*100:.2f}%)\n\n")
    
    f.write(f"Class Imbalance: {sv_imbalance_ratio:.2f}:1\n")
    if sv_imbalance_ratio > 3:
        f.write("  Status: SIGNIFICANT IMBALANCE - requires class weights or SMOTE\n\n")
    elif sv_imbalance_ratio > 2:
        f.write("  Status: MODERATE IMBALANCE - consider class weights\n\n")
    else:
        f.write("  Status: REASONABLE BALANCE - no special handling needed\n\n")
    
    f.write("Primary Metric: Recall (Sensitivity)\n")
    f.write("  Target: 0.90+ (Good), 0.95+ (Excellent)\n")
    f.write("  Rationale: Missing high-risk SVs is costly\n\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("TARGET VALIDATION\n")
    f.write("="*80 + "\n")
    f.write(f"Overlapping labels: {total_overlap:,} (should be 0)\n")
    f.write(f"Unlabeled variants: {unlabeled:,}\n")
    f.write("\nValidation: PASS" if total_overlap == 0 else "\nValidation: WARNING - overlaps detected\n")
    
    f.write("\n\n" + "="*80 + "\n")
    f.write("NEXT STEPS\n")
    f.write("="*80 + "\n")
    f.write("1. Proceed to Phase 2.2: Feature Quality Checks\n")
    f.write("2. Remove features with >80% missing values\n")
    f.write("3. Identify zero-variance features\n")
    f.write("4. Validate data types after PostgreSQL conversion\n")

print(f"\nReport saved: {report_path}")
print("\n" + "="*80)
print("PHASE 2.1 COMPLETE - Target Definition")
print("="*80)
print(f"\nGenerated {len(list(FIGURES_DIR.glob('*.png')))} visualizations")
print(f"Reports: {REPORTS_DIR}")
print("\nTargets Defined:")
print(f"  1. Variant Pathogenicity: {usable_for_training:,} training samples")
print(f"  2. SV Risk: {total_sv:,} training samples")
print("\nNext: Phase 2.2 - Feature Quality Checks")